# Regime-Switching VAR

Fit MS-VAR and visualize regime probabilities and conditional coefficients.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from models.ms_var import MarkovSwitchingVAR

rng = np.random.default_rng(42)
T = 400
n_vars = 3
data = np.zeros((T, n_vars))
regime = 0
labels = np.zeros(T, dtype=int)

for t in range(1, T):
    if regime == 0 and rng.random() < 0.03:
        regime = 1
    elif regime == 1 and rng.random() < 0.08:
        regime = 0
    labels[t] = regime
    vol = 0.2 if regime == 0 else 0.8
    coef = 0.1 if regime == 0 else 0.5
    data[t, 0] = coef * data[t-1, 0] + vol * rng.standard_normal()
    data[t, 1] = 0.4 * data[t-1, 0] + vol * rng.standard_normal()
    data[t, 2] = 0.3 * data[t-1, 1] + vol * rng.standard_normal()

print(f'Data: {data.shape}, Regime 0: {(labels==0).sum()}, Regime 1: {(labels==1).sum()}')

In [ ]:
model = MarkovSwitchingVAR(n_vars=n_vars, n_lags=1, n_regimes=2)
model.fit(data, max_iter=30)
print(f'Log-likelihood: {model.log_likelihood:.2f}')
print(f'Current regime: {model.get_current_regime()}')

In [ ]:
# Plot smoothed regime probabilities
probs = model.smoothed_probs
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(probs[:, 0], label='Regime 0 (Low Vol)', color='blue', alpha=0.7)
axes[0].plot(probs[:, 1], label='Regime 1 (High Vol)', color='red', alpha=0.7)
axes[0].legend()
axes[0].set_title('Smoothed Regime Probabilities')
axes[0].set_ylim(0, 1)

axes[1].plot(labels[1:], label='True Regime', color='green', alpha=0.5)
axes[1].set_title('True Regime Labels')

plt.tight_layout()
plt.show()